## Building basic likelihoods

In [1]:
import sys
import os
import glob
import importlib

import tqdm

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pp

%matplotlib inline

In [3]:
import jax
jax.config.update('jax_enable_x64', True)

import jax.random
import jax.numpy as jnp

In [4]:
import discovery as ds
import discovery.models.nanograv as ds_nanograv
import discovery.samplers.numpyro as ds_numpyro
importlib.reload(ds)
importlib.reload(ds_nanograv)
importlib.reload(ds_numpyro)
importlib.reload(ds.pulsar)

/pulsar/psr/software/stable/src/util/miniforge/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'discovery.pulsar' from '/h/mit227/discovery-ppa/src/discovery/pulsar.py'>

Read nanograv pulsars

In [5]:
allpsrs = [ds.Pulsar.read_feather(psrfile) for psrfile in sorted(glob.glob('../data/*-[JB]*.feather'))]

In [6]:
psr = allpsrs[0]
print(type(psr))
print(len(psr.toas))
print(psr.toas[:35])
# print(psr.stoas[:35])
print(psr.toaerrs[:35])
print(psr.residuals[:35])
print(psr.freqs[:35])

<class 'discovery.pulsar.Pulsar'>
7758
[4.61019373e+09 4.61019373e+09 4.61019373e+09 4.61019373e+09
 4.61019373e+09 4.61019373e+09 4.61019373e+09 4.61019373e+09
 4.61019373e+09 4.61019373e+09 4.61019373e+09 4.61019373e+09
 4.61019373e+09 4.61019373e+09 4.61019373e+09 4.61019373e+09
 4.61019373e+09 4.61019373e+09 4.61019373e+09 4.61019373e+09
 4.61019373e+09 4.61019505e+09 4.61019505e+09 4.61019505e+09
 4.61019505e+09 4.61019505e+09 4.61019505e+09 4.61019505e+09
 4.61019505e+09 4.61019505e+09 4.61019505e+09 4.61019505e+09
 4.61019505e+09 4.61019505e+09 4.61019505e+09]
[1.2120e-06 1.0176e-05 1.7050e-06 9.5200e-07 1.9680e-06 1.2720e-06
 2.8620e-06 4.6600e-07 3.1000e-07 5.2400e-07 5.2300e-07 1.2310e-06
 6.1100e-07 2.0800e-06 3.8100e-07 2.1510e-06 4.9800e-07 5.5400e-07
 2.1140e-06 6.2300e-07 4.6570e-06 1.0958e-05 6.0000e-06 1.9960e-06
 3.8360e-06 9.4090e-06 5.0850e-06 2.7160e-06 1.7750e-06 8.9900e-07
 4.9100e-07 5.9500e-07 3.0490e-06 8.5800e-07 5.8900e-07]
[-1.57059796e-06 -1.71515979e-06 -

In [7]:
psr.Mmat

array([[ 5.36210047e-03,  1.31863846e+06, -1.62138644e+14, ...,
         3.26448988e-01,  1.06568942e-01,  3.47893232e-02],
       [ 5.36210047e-03,  1.31863846e+06, -1.62138644e+14, ...,
         3.43616792e-01,  1.18072499e-01,  4.05716934e-02],
       [ 5.36210047e-03,  1.31863846e+06, -1.62138644e+14, ...,
         3.52091419e-01,  1.23968367e-01,  4.36481982e-02],
       ...,
       [ 5.36210047e-03, -1.31858947e+06, -1.62126595e+14, ...,
         5.15706321e-01,  2.65953009e-01,  1.37153648e-01],
       [ 5.36210047e-03, -1.31858947e+06, -1.62126595e+14, ...,
         5.59434253e-01,  3.12966684e-01,  1.75084283e-01],
       [ 5.36210047e-03, -1.31858947e+06, -1.62126595e+14, ...,
         5.30549118e-01,  2.81482367e-01,  1.49340222e-01]],
      shape=(7758, 166))

In [8]:
psr

<Pulsar B1855+09: 7758 res, 166 pars>

In [9]:
print(psr.name)
print(psr.pos)
print(np.linalg.norm(psr.pos))
print(psr.flags)

B1855+09
[0.23527600482851654, -0.957353115974499, 0.1676908252882342]
0.9999999999999999
{'format': array(['Tempo2', 'Tempo2', 'Tempo2', ..., 'Tempo2', 'Tempo2', 'Tempo2'],
      shape=(7758,), dtype='<U6'), 'name': array(['53358.000051.3.000.000.15y.x.ff',
       '53358.000051.3.000.000.15y.x.ff',
       '53358.000051.3.000.000.15y.x.ff', ...,
       'puppi_59051_B1855+09_0749.15y.x.ff',
       'puppi_59051_B1855+09_0749.15y.x.ff',
       'puppi_59051_B1855+09_0749.15y.x.ff'], shape=(7758,), dtype='<U38'), 'fe': array(['L-wide', 'L-wide', 'L-wide', ..., 'L-wide', 'L-wide', 'L-wide'],
      shape=(7758,), dtype='<U6'), 'be': array(['ASP', 'ASP', 'ASP', ..., 'PUPPI', 'PUPPI', 'PUPPI'],
      shape=(7758,), dtype='<U5'), 'f': array(['L-wide_ASP', 'L-wide_ASP', 'L-wide_ASP', ..., 'L-wide_PUPPI',
       'L-wide_PUPPI', 'L-wide_PUPPI'], shape=(7758,), dtype='<U12'), 'bw': array(['4', '4', '4', ..., '12.5', '12.5', '12.5'],
      shape=(7758,), dtype='<U5'), 'tobs': array(['903.76', '903.76

In [10]:
allpsrs_chain = [ds.pulsar.read_chain(psrfile) for psrfile in sorted(glob.glob('../data/*-[JB]*.feather'))]

In [11]:
psr_chain = allpsrs_chain[0]
psr_chain

,toas,stoas,toaerrs,residuals,freqs,backend_flags,Mmat_0,Mmat_1,Mmat_2,Mmat_3,...,flags_wt,flags_flux,flags_fluxe,flags_proc,flags_pta,flags_ver,flags_to,flags_pout_gibbs,flags_clkcorr,flags_simul
0,4.610194e+09,4.610194e+09,1.212000e-06,-1.570598e-06,1386.037543,L-wide_ASP,0.005362,1.318638e+06,-1.621386e+14,-1.0,...,2.6903,3.25644,0.021,15y,NANOGrav,2021.08.25-9d8d617,-0.839e-6,0.012491908440860854,2.630754008821355e-05,
1,4.610194e+09,4.610194e+09,1.017600e-05,-1.715160e-06,1410.038193,L-wide_ASP,0.005362,1.318638e+06,-1.621386e+14,-1.0,...,1.086,1.02708,0.053,15y,NANOGrav,2021.08.25-9d8d617,-0.839e-6,0.012905991732230643,2.6307540088161908e-05,
2,4.610194e+09,4.610194e+09,1.705000e-06,-8.360033e-07,1422.038518,L-wide_ASP,0.005362,1.318638e+06,-1.621386e+14,-1.0,...,2.5761,2.44634,0.023,15y,NANOGrav,2021.08.25-9d8d617,-0.839e-6,0.012457192575780427,2.630754008813714e-05,
3,4.610194e+09,4.610194e+09,9.520000e-07,-5.140002e-07,1414.038302,L-wide_ASP,0.005362,1.318638e+06,-1.621386e+14,-1.0,...,2.8314,4.10634,0.022,15y,NANOGrav,2021.08.25-9d8d617,-0.839e-6,0.01234654947538391,2.630754008815364e-05,
4,4.610194e+09,4.610194e+09,1.968000e-06,-2.606156e-08,1402.037977,L-wide_ASP,0.005362,1.318638e+06,-1.621386e+14,-1.0,...,2.6525,2.10044,0.022,15y,NANOGrav,2021.08.25-9d8d617,-0.839e-6,0.01288848991589095,2.6307540088178913e-05,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7753,5.102021e+09,5.102021e+09,2.059000e-06,8.940844e-07,1722.979881,L-wide_PUPPI,0.005362,-1.318589e+06,-1.621266e+14,-1.0,...,15.068,0.673438,0.0079,15y,NANOGrav,2021.08.25-9d8d617,,0.01296979848578532,2.776931610305224e-05,
7754,5.102021e+09,5.102021e+09,6.560000e-07,1.781750e-06,1686.731252,L-wide_PUPPI,0.005362,-1.318589e+06,-1.621266e+14,-1.0,...,11.499,3.57581,0.013,15y,NANOGrav,2021.08.25-9d8d617,,0.012244980649401821,2.776931610308031e-05,
7755,5.102021e+09,5.102021e+09,1.982000e-06,2.220510e-06,1674.821045,L-wide_PUPPI,0.005362,-1.318589e+06,-1.621266e+14,-1.0,...,10.409,1.12112,0.013,15y,NANOGrav,2021.08.25-9d8d617,,0.012503069874448748,2.7769316103089936e-05,
7756,5.102021e+09,5.102021e+09,6.450000e-07,1.934482e-06,1749.682344,L-wide_PUPPI,0.005362,-1.318589e+06,-1.621266e+14,-1.0,...,8.843,4.59824,0.017,15y,NANOGrav,2021.08.25-9d8d617,,0.012712839123538945,2.7769316103032728e-05,


### Single pulsar likelihood

#### Measurement noise only, no backends

In [12]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement_simple(psr)])

What are the active parameters?

In [13]:
m.logL.params

['B1855+09_efac', 'B1855+09_log10_t2equad']

Sample random values from their priors

In [14]:
p0 = ds.sample_uniform(m.logL.params); p0

{'B1855+09_efac': 8.832931685975622,
 'B1855+09_log10_t2equad': -5.6039813010511965}

Evaluate the likelihood

In [15]:
m.logL(p0)

Array(81326.34940298, dtype=float64)

Try compiled version, grad

In [16]:
jax.jit(m.logL)(p0)

Array(81326.34940298, dtype=float64)

In [17]:
jax.grad(m.logL)(p0)

{'B1855+09_efac': Array(-873.36558031, dtype=float64, weak_type=True),
 'B1855+09_log10_t2equad': Array(-12680.86970511, dtype=float64, weak_type=True)}

#### Measurement noise only, nanograv backends, free parameters

In [18]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr)])

In [19]:
m.logL.params

['B1855+09_430_ASP_efac',
 'B1855+09_430_PUPPI_efac',
 'B1855+09_L-wide_ASP_efac',
 'B1855+09_L-wide_PUPPI_efac',
 'B1855+09_430_ASP_log10_t2equad',
 'B1855+09_430_PUPPI_log10_t2equad',
 'B1855+09_L-wide_ASP_log10_t2equad',
 'B1855+09_L-wide_PUPPI_log10_t2equad']

In [20]:
m.logL(ds.sample_uniform(m.logL.params))

Array(88539.01361682, dtype=float64)

#### Measurement noise only, nanograv backends, parameters from noisedict

In [21]:
psr.noisedict

{'B1855+09_430_ASP_efac': 1.115935306813982,
 'B1855+09_430_ASP_log10_t2equad': -7.564164330699591,
 'B1855+09_430_PUPPI_efac': 1.000049037085653,
 'B1855+09_430_PUPPI_log10_t2equad': -6.572540211467256,
 'B1855+09_L-wide_ASP_efac': 1.043114017270374,
 'B1855+09_L-wide_ASP_log10_t2equad': -6.517929916655293,
 'B1855+09_L-wide_PUPPI_efac': 1.1118432332882,
 'B1855+09_L-wide_PUPPI_log10_t2equad': -7.755603780476984,
 'B1855+09_430_ASP_log10_ecorr': -6.798122106550257,
 'B1855+09_430_PUPPI_log10_ecorr': -5.6989064141929715,
 'B1855+09_L-wide_ASP_log10_ecorr': -6.120457109433745,
 'B1855+09_L-wide_PUPPI_log10_ecorr': -6.641667916624413,
 'B1855+09_red_noise_log10_A': -13.940953818371378,
 'B1855+09_red_noise_gamma': -3.68432133461766}

In [22]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict)])

In [23]:
m.logL.params

[]

In [24]:
m.logL(ds.sample_uniform(m.logL.params))

np.float64(96686.81011373615)

#### Add ECORR noise (GP), free params

In [25]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr),
                         ds.makegp_ecorr(psr)])

In [26]:
m.logL.params

['B1855+09_430_ASP_efac',
 'B1855+09_430_ASP_log10_ecorr',
 'B1855+09_430_ASP_log10_t2equad',
 'B1855+09_430_PUPPI_efac',
 'B1855+09_430_PUPPI_log10_ecorr',
 'B1855+09_430_PUPPI_log10_t2equad',
 'B1855+09_L-wide_ASP_efac',
 'B1855+09_L-wide_ASP_log10_ecorr',
 'B1855+09_L-wide_ASP_log10_t2equad',
 'B1855+09_L-wide_PUPPI_efac',
 'B1855+09_L-wide_PUPPI_log10_ecorr',
 'B1855+09_L-wide_PUPPI_log10_t2equad']

In [27]:
m.logL(ds.sample_uniform(m.logL.params))

Array(92845.71497077, dtype=float64)

#### Add ECORR noise (GP), noisedict params

In [28]:
print(psr.noisedict)

{'B1855+09_430_ASP_efac': 1.115935306813982, 'B1855+09_430_ASP_log10_t2equad': -7.564164330699591, 'B1855+09_430_PUPPI_efac': 1.000049037085653, 'B1855+09_430_PUPPI_log10_t2equad': -6.572540211467256, 'B1855+09_L-wide_ASP_efac': 1.043114017270374, 'B1855+09_L-wide_ASP_log10_t2equad': -6.517929916655293, 'B1855+09_L-wide_PUPPI_efac': 1.1118432332882, 'B1855+09_L-wide_PUPPI_log10_t2equad': -7.755603780476984, 'B1855+09_430_ASP_log10_ecorr': -6.798122106550257, 'B1855+09_430_PUPPI_log10_ecorr': -5.6989064141929715, 'B1855+09_L-wide_ASP_log10_ecorr': -6.120457109433745, 'B1855+09_L-wide_PUPPI_log10_ecorr': -6.641667916624413, 'B1855+09_red_noise_log10_A': -13.940953818371378, 'B1855+09_red_noise_gamma': -3.68432133461766}


In [33]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict),
                         ds.makegp_ecorr(psr, psr.noisedict)])

In [34]:
m.logL.params

[]

In [35]:
m.logL(ds.sample_uniform(m.logL.params))

np.float64(103037.9457516601)

#### Add timing model

In [36]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict),
                         ds.makegp_ecorr(psr, psr.noisedict),
                         ds.makegp_timing(psr, svd=True)])

In [38]:
m.logL.params

[]

In [39]:
m.logL(ds.sample_uniform(m.logL.params))

np.float64(94817.46786020289)

#### Add red noise (powerlaw)

In [51]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict),
                         ds.makegp_ecorr(psr, psr.noisedict),
                         ds.makegp_timing(psr, svd=True),
                         ds.makegp_fourier(psr, ds.powerlaw, components=30, name='rednoise')])

In [52]:
m.logL.params

['B1855+09_rednoise_log10_A', 'B1855+09_rednoise_gamma']

In [53]:
m.logL(ds.sample_uniform(m.logL.params))

Array(94817.46858283, dtype=float64)

#### Add red noise (powerlaw, fixed gamma)

In [54]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict),
                         ds.makegp_ecorr(psr, psr.noisedict),
                         ds.makegp_timing(psr, svd=True),
                         ds.makegp_fourier(psr, ds.partial(ds.powerlaw, gamma=4.33), components=30, name='rednoise')])

In [55]:
m.logL.params

['B1855+09_rednoise_log10_A']

In [56]:
m.logL(ds.sample_uniform(m.logL.params))

Array(94818.95650736, dtype=float64)

#### Add red noise (free spectrum)

In [ ]:
m = ds.PulsarLikelihood([psr.residuals,
                         ds.makenoise_measurement(psr, psr.noisedict),
                         ds.makegp_ecorr(psr, psr.noisedict),
                         ds.makegp_timing(psr, svd=True),
                         ds.makegp_fourier(psr, ds.freespectrum, components=30, name='rednoise')])

In [59]:
m.logL.params

['B1855+09_rednoise_log10_rho(30)']

In [64]:
m.logL({'B1855+09_rednoise_log10_rho(30)': 1e-6 * np.random.randn(30)})

Array(94024.38714473, dtype=float64)

### Multiple pulsars

In [69]:
psrs = allpsrs[:3]

#### Combined likelihood

In [ ]:
m = ds.ArrayLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True),
                                             ds.makegp_fourier(psr, ds.powerlaw, components=30, name='rednoise')])
                        for psr in psrs])

In [78]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A']

In [79]:
m.logL(ds.sample_uniform(m.logL.params))

Array(442105.13787401, dtype=float64)

#### Add common noise

Indicating parameters under common shares them among pulsars

In [ ]:
T = ds.getspan(psrs)

In [115]:
m = ds.ArrayLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True),
                                             ds.makegp_fourier(psr, ds.powerlaw, components=30, T=T, name='rednoise'),
                                             ds.makegp_fourier(psr, ds.powerlaw, components=14, T=T, name='crn',
                                                               common=['crn_log10_A', 'crn_gamma'])])
                        for psr in psrs])

In [116]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A',
 'crn_gamma',
 'crn_log10_A']

In [117]:
p0 = ds.sample_uniform(m.logL.params)
m.logL(p0)

Array(487234.62649703, dtype=float64)

#### Parallelize red components

Coordinated timespan is required

In [118]:
m = ds.ArrayLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True)]) for psr in psrs],
                       commongp = [ds.makecommongp_fourier(psrs, ds.powerlaw, components=30, T=T, name='rednoise'),
                                   ds.makecommongp_fourier(psrs, ds.powerlaw, components=14, T=T, name='crn',
                                                           common=['crn_log10_A', 'crn_gamma'])])

In [119]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A',
 'crn_gamma',
 'crn_log10_A']

In [120]:
m.logL(p0)

Array(487234.62649703, dtype=float64)

#### Reuse Fourier vectors

`ds.makepowerlaw_crn` yields the sum of two powerlaws, with possibly different number of components.

In [95]:
m = ds.ArrayLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True)]) for psr in psrs],
                       commongp = ds.makecommongp_fourier(psrs, ds.makepowerlaw_crn(components=14), components=30, T=T, name='rednoise',
                                                          common=['crn_log10_A', 'crn_gamma']))

In [96]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A',
 'crn_gamma',
 'crn_log10_A']

In [97]:
m.logL(p0)

Array(476084.74919831, dtype=float64)

#### Add global spatially correlated process

Note `ds.makeglobalgp_fourier` requires the ORF, but not the `common` specification, which is automatic.

In [124]:
m = ds.ArrayLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True)]) for psr in psrs],
                       commongp = ds.makecommongp_fourier(psrs, ds.powerlaw, components=30, T=T, name='rednoise'),
                       globalgp = ds.makeglobalgp_fourier(psrs, ds.powerlaw, ds.hd_orf, components=14, T=T, name='gw'))

In [125]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A',
 'gw_gamma',
 'gw_log10_A']

In [126]:
p0 = ds.sample_uniform(m.logL.params)
m.logL(p0)

Array(466332.09973769, dtype=float64)

#### Another way of doing this (useful if variable GPs differ among pulsars)

In [127]:
m = ds.GlobalLikelihood([ds.PulsarLikelihood([psr.residuals,
                                             ds.makenoise_measurement(psr, psr.noisedict),
                                             ds.makegp_ecorr(psr, psr.noisedict),
                                             ds.makegp_timing(psr, svd=True),
                                             ds.makegp_fourier(psr, ds.powerlaw, components=30, name='rednoise')]) for psr in psrs],
                        globalgp = ds.makeglobalgp_fourier(psrs, ds.powerlaw, ds.hd_orf, components=14, T=T, name='gw'))

In [128]:
m.logL.params

['B1855+09_rednoise_gamma',
 'B1855+09_rednoise_log10_A',
 'B1937+21_rednoise_gamma',
 'B1937+21_rednoise_log10_A',
 'B1953+29_rednoise_gamma',
 'B1953+29_rednoise_log10_A',
 'gw_gamma',
 'gw_log10_A']

In [129]:
m.logL(p0)

Array(466332.19059264, dtype=float64)